# se-path-multi-v1

Practice, revision and quiz actions. Coding stays masked. No `prior_strength` slot.
The served policy is a native `sb3_contrib.MaskablePPO` zip (`maskable_ppo_final.zip`), not `policy.npz`.
Keep `RUN_FULL_TRAINING = False` until the intentional Colab run.
Return is the sum of simulator rewards. Do not call it accuracy.

Run the cells from the top. The notebook installs dependencies, uploads `se_path_multi_v1_colab.zip`, and finds the source root inside that archive. It does not expect the source to already be at `/content/c4`.


In [ ]:
%pip install -q numpy gymnasium==1.4.0 stable-baselines3==2.9.0 sb3-contrib==2.9.0


In [ ]:
RUN_FULL_TRAINING = False
print('RUN_FULL_TRAINING', RUN_FULL_TRAINING)


In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

MARKER = Path('training_se_v2') / 'multi_action' / 'smoke_runner.py'

def find_zip() -> Path:
    found = sorted(Path('/content').glob('se_path_multi_v1_colab.zip'))
    if found:
        return found[0]
    from google.colab import files
    uploaded = files.upload()
    zips = [Path('/content') / name for name in uploaded if name.endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Upload se_path_multi_v1_colab.zip')
    return zips[0]

def find_source_root(start: Path) -> Path:
    if (start / MARKER).is_file():
        return start
    for path in start.rglob('smoke_runner.py'):
        if path.parent.name == 'multi_action' and path.parent.parent.name == 'training_se_v2':
            return path.parents[2]
    raise FileNotFoundError(f'{MARKER} was not found in the extracted ZIP')

zip_path = find_zip()
extract_to = Path('/content/se_path_multi_src')
if extract_to.exists():
    shutil.rmtree(extract_to)
extract_to.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_to)
ROOT = find_source_root(extract_to)
protocol_file = ROOT / 'training_se_v1' / 'protocol.py'
if not protocol_file.is_file():
    raise FileNotFoundError(protocol_file)
sys.path[:] = [str(ROOT)] + [entry for entry in sys.path if entry not in ('', str(ROOT))]
print('ZIP', zip_path)
print('ROOT', ROOT)
print('protocol', protocol_file)


In [ ]:
import training_se_v1.protocol as se_protocol
from training_se_v2.multi_action.smoke_runner import main
print('protocol module', se_protocol.__file__)
if not str(se_protocol.__file__).startswith(str(ROOT)):
    raise RuntimeError(f'protocol resolved outside the extracted ZIP: {se_protocol.__file__}')
main()


In [ ]:
if 'ROOT' not in dir():
    raise RuntimeError('Run the ZIP extraction cell first.')
if not RUN_FULL_TRAINING:
    print('FULL TRAINING SKIPPED. Smoke above does not activate a policy.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    from training_se_v2.multi_action.full_run import run_full
    out = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v1/run_full')
    manifest = run_full(out, timesteps=80_000, episodes=20)
    print(manifest['algorithm'], manifest['training_mode'], 'seed', manifest['selected_seed'])
    print('export', out / 'export')


## What the full cell does

When `RUN_FULL_TRAINING` is True, `run_full` trains `sb3_contrib.MaskablePPO` with `learn(..., use_masking=True)` for seeds 0, 1 and 2.
Evaluation calls `predict(..., action_masks=mask, deterministic=True)`.
Baselines on the same masks: `weakest_practice`, `recent_performance`, `random_valid`.
The checkpoint is chosen by mean Y3S1+Y3S2 validation return. The held-out test is not used for that choice.

Export directory `.../run_full/export/` contains:

- `maskable_ppo_final.zip` (native selected policy)
- `models/seed_{0,1,2}/maskable_ppo_final.zip`
- `manifest.json` (`training_mode=full`, `algorithm=maskable_ppo`, `activation_allowed=true`)
- `observation_schema.json`, `selection_val.json`, `final_test.json`
- `train_config.json`, `dependency_versions.json`, `checksums.json`

Copy that `export` folder to `ai-services/c4/runtime-artifacts/se_path_multi_v1/active/` and restart the C4 Python service.
Do not set `activation_allowed` by hand, and do not copy `experiments/parity_runs` or `experiments/smoke_runs`.
Node calls `/v1/se-path-multi-v1/predict` on the next plan build. There is no separate Node activation switch.
Restart Node once so it loads this integration. After that, Python restart plus a valid export is enough.
